In [12]:
!pip install pandas

import pandas as pd
from google.colab import files

In [13]:
data = {
    "device_id":[101,101,102,102,103,103,104,104],
    "device_name":[
        "AC Unit","AC Unit",
        "Server Rack","Server Rack",
        "LED Lights","LED Lights",
        "Projector","Projector"
    ],
    "date":[
        "2026-06-01",
        "2026-06-02",
        "2026-06-01",
        "2026-06-02",
        "2026-06-01",
        "2026-06-02",
        "2026-06-01",
        "2026-06-02"
    ],
    "energy_kwh":[
        8.5,
        9.2,
        25.5,
        24.8,
        4.0,
        3.5,
        12.2,
        6.8
    ]
}

df = pd.DataFrame(data)

df.to_csv("sensor_logs.csv", index=False)

print("sensor_logs.csv created successfully!\n")

sensor_logs.csv created successfully!



In [14]:
df = pd.read_csv("sensor_logs.csv")

In [15]:
df["date"] = pd.to_datetime(df["date"])
df["energy_kwh"] = pd.to_numeric(df["energy_kwh"], errors="coerce")
df = df.dropna()

In [16]:
daily_summary = df.groupby(
    ["date","device_id","device_name"]
)["energy_kwh"].sum().reset_index()

daily_summary.rename(
    columns={"energy_kwh":"Daily_Usage_kWh"},
    inplace=True
)

In [17]:
df["week"] = df["date"].dt.isocalendar().week

weekly_summary = df.groupby(
    ["week","device_id","device_name"]
)["energy_kwh"].sum().reset_index()

weekly_summary.rename(
    columns={"energy_kwh":"Weekly_Usage_kWh"},
    inplace=True
)

In [18]:
THRESHOLD = 10

daily_summary["Alert"] = daily_summary["Daily_Usage_kWh"].apply(
    lambda x: "High Usage" if x > THRESHOLD else "Normal"
)

print("========== EXECUTION LOG ==========\n")

for _, row in daily_summary.iterrows():
    if row["Daily_Usage_kWh"] > THRESHOLD:
        print(f"ALERT: {row['device_name']} (Device {row['device_id']}) "
              f"used {row['Daily_Usage_kWh']} kWh on {row['date'].date()}")

print("\nPipeline executed successfully!")

========== EXECUTION LOG ==========

ALERT: Server Rack (Device 102) used 25.5 kWh on 2026-06-01
ALERT: Projector (Device 104) used 12.2 kWh on 2026-06-01
ALERT: Server Rack (Device 102) used 24.8 kWh on 2026-06-02

Pipeline executed successfully!


In [19]:
daily_summary.to_csv("daily_energy_report.csv", index=False)
weekly_summary.to_csv("weekly_energy_report.csv", index=False)

print("\nReports generated successfully!")



Reports generated successfully!


In [20]:
print("\nDaily Energy Report")
print(daily_summary)

print("\nWeekly Energy Report")
print(weekly_summary)



Daily Energy Report
        date  device_id  device_name  Daily_Usage_kWh       Alert
0 2026-06-01        101      AC Unit              8.5      Normal
1 2026-06-01        102  Server Rack             25.5  High Usage
2 2026-06-01        103   LED Lights              4.0      Normal
3 2026-06-01        104    Projector             12.2  High Usage
4 2026-06-02        101      AC Unit              9.2      Normal
5 2026-06-02        102  Server Rack             24.8  High Usage
6 2026-06-02        103   LED Lights              3.5      Normal
7 2026-06-02        104    Projector              6.8      Normal

Weekly Energy Report
   week  device_id  device_name  Weekly_Usage_kWh
0    23        101      AC Unit              17.7
1    23        102  Server Rack              50.3
2    23        103   LED Lights               7.5
3    23        104    Projector              19.0


In [22]:
files.download("daily_energy_report.csv")
files.download("weekly_energy_report.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>